# 31. 큰 그룹 보조 학습: 신장(KIPAN·KIRC)·뇌(GBMLGG·LGG)를 묶어 학습 (H31)

## 배경 (30번 결과)
- C 시드 평균 OOF에서 KIRC·KIPAN 정답률 19%대, 오답의 대부분은 **같은 그룹 안 혼동** (KIRC → KIPAN 53%, KIPAN → KIRC 45%, GBMLGG → LGG 44%)
- **그룹 밖 오답**은 신장 32.7%, 뇌 24.1% (신장 → PRAD 65명 등)
- 지금은 신장 환자 849명이 KIRC(334)·KIPAN(515) 두 클래스로 나뉘어 학습 → "신장"이라는 공통 신호를 둘로 나눠 배우는 구조

## 가설
- H31a: 신장·뇌를 큰 그룹으로 묶어 학습하면 "신장인지 아닌지"를 더 확실하게 배워 **그룹 밖 오답이 줄고** C보다 오른다 (그룹 안 비율은 C 그대로)
- H31b: 여기에 그룹 안 모델(KIRC vs KIPAN, LGG vs GBMLGG)을 더하면 그룹 안 혼동도 준다
- H31c (31a 결과 후 추가): 31a는 balanced weight 때문에 신장 그룹 전체 무게가 C의 약 절반이 되어 신장을 덜 예측함 → **큰 그룹 모델의 샘플 가중치를 원래 26개 라벨 기준 balanced로** 주면 신장 전체 무게가 C와 같아져 그룹 밖 오답이 준다

## 선영님 29번 리뷰 반영
1. **같은 프로필 착시**: 같은 환자가 KIRC·KIPAN으로 한 번씩 들어 있고, 공통 CV에서는 이 묶음의 81%가 다른 fold로 갈라짐 → 그룹 안 모델이 공통 CV에서만 오르는 착시 (29번: 공통 CV +0.0079, 5/5 → 같은 프로필 묶음 CV −0.0033, 0/5)
   - → **두 가지 검증을 모두** 봄: 공통 CV + 같은 프로필을 한 fold에 묶는 CV (29번과 같은 `StratifiedGroupKFold`)
   - → 효과가 어디서 나오는지 보이게 **31a(큰 그룹 + 그룹 안은 C 비율)** 와 **31b(31a + 그룹 안 모델)** 를 따로 비교
2. **다른 암종이 끌려오는지**: C에서 다른 암종인데 신장·뇌로 예측되는 환자가 254명 → 그룹을 합치면 더 끌려올 수 있어 **나머지 22개 암종 F1**과 **그룹으로 끌려온 환자 수**를 같이 확인

## 방법 (모두 fold 학습 부분에서만 학습, test는 예측만)
- **큰 그룹 모델**: 22b 피처 XGB로 24개 분류 (신장 = KIPAN + KIRC, 뇌 = GBMLGG + LGG, 나머지 22개 그대로)
- **31a**: XGB 쪽 26개 확률 = 큰 그룹 확률 × (22b XGB의 그룹 안 비율) → XGB 0.7 + LR(any) 0.3 섞기 (C와 같은 방식, XGB의 그룹 확률만 바뀜)
- **31b**: XGB 쪽 26개 확률 = 큰 그룹 확률 × (그룹 안 이진 XGB 확률, 그 그룹 학습 환자만으로 학습) → 같은 섞기
- **31c**: 31a와 같고, 큰 그룹 XGB의 샘플 가중치만 26개 라벨 기준 balanced (라벨은 24개로 학습, KIRC 환자는 KIRC 몫·KIPAN 환자는 KIPAN 몫의 무게) → train 라벨로만 정하는 가중치
- 비교 대상: C(22b XGB 0.7 + LR 0.3, LB 0.3650), 같은 fold 짝 비교, 채택 기준 4/5
- 시드 평균 5개(팀 규칙)는 **두 검증 모두에서 C보다 4/5 이상**인 설정만 실행
- 쓰는 정보: train 라벨 + TCGA 코호트 명명(도메인 지식) → Data Leakage 아님, test 후처리 아님
- 실행 환경: macOS M1, 제출 파일은 Windows에서만 저장

#### 준비: 데이터, 22b 피처, 그룹 정의, 두 가지 검증 분할
- 같은 프로필 묶음: 유전자 4,384칸 변이 문자열이 완전히 같은 행끼리 같은 그룹 (29번과 같은 방식)

In [1]:
import sys; sys.path.append('..')
import time, platform, numpy as np, pandas as pd, xgboost as xgb
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED, WT
from src.data import load_data, get_gene_cols
from src.features import mutation_features, mutation_type, aa_substitution
from src.validation import seed_everything, encode_target, get_folds, run_cv, save_submission, macro_f1

print(f'{platform.system()} | CPU {platform.machine()} | xgboost {xgb.__version__}')
seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
ci = {c: i for i, c in enumerate(le.classes_)}
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

class WeightedXGB(xgb.XGBClassifier):   # 31c: 밖에서 준 sample_weight 그대로 사용
    pass

BEST = dict(max_depth=3, learning_rate=0.1, n_estimators=200, colsample_bytree=0.3,
            tree_method='hist', eval_metric='mlogloss', random_state=SEED, n_jobs=-1)

def sub_ratio_matrix(df):
    rows = []
    for row in df[genes].to_numpy():
        subs = (aa_substitution(t) for v in row if v != WT for t in v.split() if mutation_type(t) == 'missense')
        c = Counter(s for s in subs if s is not None)
        tot = sum(c.values())
        rows.append({k: n / tot for k, n in c.items()} if tot else {})
    M = pd.DataFrame(rows, index=df.index).fillna(0).astype(np.float32)
    M.columns = ['sub__' + k.replace('>', '_') for k in M.columns]
    return M

class TopNSubs(BaseEstimator, TransformerMixin):
    def __init__(self, n=100):
        self.n = n
    def fit(self, X, y=None):
        sub = [c for c in X.columns if c.startswith('sub__')]
        keep = set(X[sub].sum().sort_values(ascending=False).index[:self.n])
        self.drop_ = [c for c in sub if c not in keep]
        return self
    def transform(self, X):
        return X.drop(columns=self.drop_)

S_tr = sub_ratio_matrix(train)
S_te = sub_ratio_matrix(test).reindex(columns=S_tr.columns, fill_value=0)
X22b, X22b_te = pd.concat([F_tr['any'], F_tr['ratio'], S_tr], axis=1), pd.concat([F_te['any'], F_te['ratio'], S_te], axis=1)

# 큰 그룹 정의 (train 라벨 + TCGA 코호트 명명), 순서 = (묶음 코호트, 세부 암종)
GROUPS = {'KIDNEY': ['KIPAN', 'KIRC'], 'BRAIN': ['GBMLGG', 'LGG']}
FOUR = ['KIPAN', 'KIRC', 'GBMLGG', 'LGG']
to_group = {c: g for g, cs in GROUPS.items() for c in cs}
true_lab = le.classes_[y]
yg_lab = np.array([to_group.get(c, c) for c in true_lab])
le_g = LabelEncoder().fit(yg_lab)
yg = le_g.transform(yg_lab)

# 같은 프로필 묶음 (29번과 같은 방식) → 두 가지 검증 분할
grp = pd.Series(pd.util.hash_pandas_object(train[genes], index=False).to_numpy()).rank(method='dense').astype(int).to_numpy()
SCHEMES = {'공통 CV': get_folds(y),
           '같은 프로필 묶음 CV': list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y, grp))}
fid = np.zeros(len(y), int)
for k, (_, va) in enumerate(SCHEMES['공통 CV']):
    fid[va] = k
g_df = pd.DataFrame({'g': grp, 'f': fid})
size = g_df.groupby('g').size(); multi = size[size > 1].index
print(f"큰 그룹 분류 {len(le_g.classes_)}개 | 신장 {int((yg_lab == 'KIDNEY').sum())}명, 뇌 {int((yg_lab == 'BRAIN').sum())}명")
print(f"같은 프로필 묶음 {len(multi)}개 중 공통 CV에서 다른 fold로 갈라진 비율: {(g_df[g_df.g.isin(multi)].groupby('g').f.nunique() > 1).mean():.1%}")

Darwin | CPU arm64 | xgboost 3.4.1
큰 그룹 분류 24개 | 신장 849명, 뇌 690명
같은 프로필 묶음 451개 중 공통 CV에서 다른 fold로 갈라진 비율: 80.7%


#### 모델 함수 (fold 학습 부분에서만 학습)
- `cv_run(folds, seed)`: 22b XGB(26개), 큰 그룹 XGB(24개), 그룹 안 이진 XGB 2개, LR을 한 번에 학습 → OOF·test 확률
- `variants(...)`: C / 31a / 31b의 26개 확률 (모두 XGB 0.7 + LR 0.3)

In [2]:
def xgb_pipe(seed, binary=False, weighted=False):
    params = {**BEST, 'random_state': seed, **({'eval_metric': 'logloss'} if binary else {})}
    return make_pipeline(TopNSubs(100), WeightedXGB(**params) if weighted else BalancedXGB(**params))

def cv_run(folds, seed=SEED, with_lr=True):
    n, nt, K = len(y), len(X22b_te), len(le.classes_)
    out = {k: [np.zeros((n, d)), np.zeros((nt, d))] for k, d in
           [('x22b', K), ('big', len(le_g.classes_)), ('big_w', len(le_g.classes_)), ('lr', K)] + [(f'in_{g}', 2) for g in GROUPS]}
    for tr, va in folds:
        Xtr, Xva = X22b.iloc[tr], X22b.iloc[va]
        w26 = compute_sample_weight('balanced', y[tr])   # 31c: 원래 26개 라벨 기준 가중치
        jobs = [('x22b', xgb_pipe(seed), Xtr, y[tr], {}), ('big', xgb_pipe(seed), Xtr, yg[tr], {}),
                ('big_w', xgb_pipe(seed, weighted=True), Xtr, yg[tr], {'weightedxgb__sample_weight': w26})]
        for g, members in GROUPS.items():
            m = np.isin(true_lab[tr], members)
            jobs.append((f'in_{g}', xgb_pipe(seed, binary=True), Xtr[m], (true_lab[tr][m] == members[1]).astype(int), {}))
        for key, mdl, Xf, yf, fkw in jobs:
            mdl.fit(Xf, yf, **fkw)
            out[key][0][va] = mdl.predict_proba(Xva)
            out[key][1] += mdl.predict_proba(X22b_te) / len(folds)
        if with_lr:
            lr = LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED)
            lr.fit(F_tr['any'].iloc[tr].astype('float32'), y[tr])
            out['lr'][0][va] = lr.predict_proba(F_tr['any'].iloc[va].astype('float32'))
            out['lr'][1] += lr.predict_proba(F_te['any'].astype('float32')) / len(folds)
    return out

def expand(big, x22b, inner=None):
    # 큰 그룹 확률(24) → 26개: 그룹 안 비율은 inner(그룹 안 모델) 또는 22b XGB 비율(inner=None)
    P = np.zeros_like(x22b)
    for c, k in ci.items():
        g = to_group.get(c)
        gi = list(le_g.classes_).index(g if g else c)
        if g is None:
            P[:, k] = big[:, gi]
            continue
        a, b = ci[GROUPS[g][0]], ci[GROUPS[g][1]]
        if inner is None:
            share = x22b[:, k] / np.maximum(x22b[:, a] + x22b[:, b], 1e-12)
        else:
            share = inner[g][:, GROUPS[g].index(c)]
        P[:, k] = big[:, gi] * share
    return P

def variants(o, lr=None):
    lr = lr if lr is not None else o['lr']
    res = {}
    for split in (0, 1):   # 0 = OOF, 1 = test
        x, big = o['x22b'][split], o['big'][split]
        inner = {g: o[f'in_{g}'][split] for g in GROUPS}
        big_w = o['big_w'][split]
        for name, xp in [('C', x), ('31a 큰 그룹 + C 비율', expand(big, x)), ('31b 큰 그룹 + 그룹 안 모델', expand(big, x, inner)),
                         ('31c 큰 그룹(26개 기준 가중치) + C 비율', expand(big_w, x))]:
            res.setdefault(name, [None, None])[split] = 0.7 * xp + 0.3 * lr[split]
    return res

#### 비교: 두 가지 검증에서 C / 31a / 31b (시드 42)
- 시간이 걸림 (검증 방식마다 XGB 5개 + LR × 5 fold, 약 25분)
- 끌려온 환자 = 실제로는 신장·뇌가 아닌데 신장·뇌 클래스로 예측된 환자 수 (C 기준 선영님 확인 254명)

In [3]:
RAW, RES = {}, {}
for sname, folds in SCHEMES.items():
    t = time.time()
    RAW[sname] = cv_run(folds)
    RES[sname] = variants(RAW[sname])
    print(f'{sname} 완료 ({time.time() - t:.0f}s)')

rest_idx = [k for c, k in ci.items() if c not in FOUR]
def metrics(po, folds, base_fs):
    pred = po.argmax(1); pl = le.classes_[pred]
    fs = np.array([macro_f1(y[va], pred[va]) for _, va in folds])
    f1 = f1_score(y, pred, average=None, labels=range(len(le.classes_)))
    row = {'CV': round(fs.mean(), 4), 'std': round(fs.std(), 4),
           'C 대비': round(fs.mean() - base_fs.mean(), 4), 'C보다 높은 fold': f'{int((fs > base_fs).sum())}/5'}
    for g, members in GROUPS.items():
        m = np.isin(true_lab, members)
        row[f'{g} 그룹 밖'] = f'{(~np.isin(pl[m], members)).mean():.1%}'
        row[f'{g} 그룹 안 혼동'] = f'{(np.isin(pl[m], members) & (pl[m] != true_lab[m])).mean():.1%}'
    row['나머지 22개 평균 F1'] = round(f1[rest_idx].mean(), 4)
    row['끌려온 환자 (신장·뇌로 예측)'] = int((~np.isin(true_lab, FOUR) & np.isin(pl, FOUR)).sum())
    return row, fs

rows, FS = [], {}
for sname, folds in SCHEMES.items():
    base_fs = np.array([macro_f1(y[va], RES[sname]['C'][0][va].argmax(1)) for _, va in folds])
    for vname, (po, _) in RES[sname].items():
        row, fs = metrics(po, folds, base_fs)
        FS[(sname, vname)] = fs
        rows.append({'검증': sname, '조합': vname, **row})
pd.DataFrame(rows)

공통 CV 완료 (516s)
같은 프로필 묶음 CV 완료 (539s)


,검증,조합,CV,std,C 대비,C보다 높은 fold,KIDNEY 그룹 밖,KIDNEY 그룹 안 혼동,BRAIN 그룹 밖,BRAIN 그룹 안 혼동,나머지 22개 평균 F1,끌려온 환자 (신장·뇌로 예측)
0,공통 CV,C,0.4557,0.0065,0.0000,0/5,32.7%,49.0%,24.6%,38.4%,0.4855,234
1,공통 CV,31a 큰 그룹 + C 비율,0.4481,0.0075,-0.0075,1/5,44.2%,41.5%,26.8%,37.7%,0.4781,109
2,공통 CV,31b 큰 그룹 + 그룹 안 모델,0.4460,0.0085,-0.0096,1/5,44.2%,42.0%,26.8%,40.6%,0.4791,104
3,공통 CV,31c 큰 그룹(26개 기준 가중치) + C 비율,0.4562,0.0091,0.0006,3/5,33.6%,48.4%,24.8%,38.3%,0.4863,224
4,같은 프로필 묶음 CV,C,0.4685,0.0156,0.0000,0/5,40.4%,26.7%,25.1%,30.1%,0.4799,232
5,같은 프로필 묶음 CV,31a 큰 그룹 + C 비율,0.4666,0.0136,-0.0018,1/5,48.5%,23.8%,28.7%,29.1%,0.4799,105
6,같은 프로필 묶음 CV,31b 큰 그룹 + 그룹 안 모델,0.4672,0.0136,-0.0013,2/5,48.4%,24.1%,29.1%,28.8%,0.4807,104
7,같은 프로필 묶음 CV,31c 큰 그룹(26개 기준 가중치) + C 비율,0.4700,0.0141,0.0016,2/5,39.3%,27.3%,26.1%,29.7%,0.4810,222


#### 네 클래스 F1 (OOF): C → 31a → 31b → 31c

In [4]:
tab = {}
for sname in SCHEMES:
    f = {v: f1_score(y, RES[sname][v][0].argmax(1), average=None, labels=range(len(le.classes_))) for v in RES[sname]}
    tab[sname] = {c: ' → '.join(f'{f[v][ci[c]]:.3f}' for v in RES[sname]) for c in FOUR}
    tab[sname]['나머지 22개 평균'] = ' → '.join(f'{f[v][rest_idx].mean():.3f}' for v in RES[sname])
pd.DataFrame(tab)

,공통 CV,같은 프로필 묶음 CV
KIPAN,0.206 → 0.163 → 0.162 → 0.201,0.323 → 0.272 → 0.287 → 0.322
KIRC,0.180 → 0.190 → 0.178 → 0.184,0.418 → 0.434 → 0.416 → 0.426
GBMLGG,0.324 → 0.314 → 0.313 → 0.324,0.389 → 0.368 → 0.384 → 0.386
LGG,0.479 → 0.484 → 0.420 → 0.483,0.578 → 0.584 → 0.568 → 0.579
나머지 22개 평균,0.485 → 0.478 → 0.479 → 0.486,0.480 → 0.480 → 0.481 → 0.481


#### 시드 평균 5개 (팀 규칙, 두 검증 모두에서 C보다 4/5 이상인 설정만)
- 17과 같은 방식: XGB만 시드 5개(42, 0, 1, 2, 3)로 학습해 확률 평균, 공통 CV fold와 LR·비율 0.7 / 0.3은 고정

In [5]:
PASS = [v for v in ['31a 큰 그룹 + C 비율', '31b 큰 그룹 + 그룹 안 모델', '31c 큰 그룹(26개 기준 가중치) + C 비율']
        if all((FS[(s, v)] > FS[(s, 'C')]).sum() >= 4 for s in SCHEMES)]
print('두 검증 모두 통과:', PASS or '없음')
SEED_RES = {}
if PASS:
    acc = {k: [RAW['공통 CV'][k][0].copy(), RAW['공통 CV'][k][1].copy()] for k in RAW['공통 CV'] if k != 'lr'}
    for s in [0, 1, 2, 3]:
        t = time.time()
        o = cv_run(SCHEMES['공통 CV'], seed=s, with_lr=False)
        for k in acc:
            acc[k][0] += o[k][0]; acc[k][1] += o[k][1]
        print(f'시드 {s} 완료 ({time.time() - t:.0f}s)')
    avg = {k: [v[0] / 5, v[1] / 5] for k, v in acc.items()}
    SEED_RES = variants(avg, lr=RAW['공통 CV']['lr'])
    folds = SCHEMES['공통 CV']
    bCs = np.array([macro_f1(y[va], SEED_RES['C'][0][va].argmax(1)) for _, va in folds])
    out = pd.DataFrame([{'조합': f'{v} 시드 평균', **metrics(SEED_RES[v][0], folds, bCs)[0]} for v in ['C'] + PASS])
else:
    out = '두 검증 모두에서 C보다 4/5 이상인 설정 없음 → 시드 평균은 실행하지 않음'
out

두 검증 모두 통과: 없음


'두 검증 모두에서 C보다 4/5 이상인 설정 없음 → 시드 평균은 실행하지 않음'

#### 제출 파일 (데스크탑에서만 저장)
- 31a·31b 시드 42판 (공통 CV 기준) + 통과한 설정의 시드 평균판, 채택 여부는 두 검증의 fold 비교로 판단

In [6]:
TAG = {'31a 큰 그룹 + C 비율': 'hier31a', '31b 큰 그룹 + 그룹 안 모델': 'hier31b', '31c 큰 그룹(26개 기준 가중치) + C 비율': 'hier31c'}
for v, tag in TAG.items():
    ups = {s: int((FS[(s, v)] > FS[(s, 'C')]).sum()) for s in SCHEMES}
    print(f"{v}: C보다 높은 fold " + ', '.join(f'{s} {u}/5' for s, u in ups.items()) +
          f" → {'채택 기준 통과' if all(u >= 4 for u in ups.values()) else '기준 미달'}")
    files = {f'blend_{tag}_0.7_LR0.3': RES['공통 CV'][v]}
    if v in SEED_RES:
        files[f'blend_{tag}_0.7_LR0.3_seed5'] = SEED_RES[v]
    if platform.system() == 'Windows':
        for fname, (po, pt) in files.items():
            save_submission(submission, pt, le, name=fname, cv_score=macro_f1(y, po.argmax(1)))   # 파일명 숫자 = OOF
if platform.system() != 'Windows':
    print(f'{platform.system()}에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)')

31a 큰 그룹 + C 비율: C보다 높은 fold 공통 CV 1/5, 같은 프로필 묶음 CV 1/5 → 기준 미달
31b 큰 그룹 + 그룹 안 모델: C보다 높은 fold 공통 CV 1/5, 같은 프로필 묶음 CV 2/5 → 기준 미달
31c 큰 그룹(26개 기준 가중치) + C 비율: C보다 높은 fold 공통 CV 3/5, 같은 프로필 묶음 CV 2/5 → 기준 미달
Darwin에서 실행 → 제출 파일 저장 안 함 (데스크탑에서 실행하면 저장)


## 결과 (macOS M1, 시드 42)

**확인**: 같은 프로필 묶음 451개 중 공통 CV에서 다른 fold로 갈라진 비율 80.7% (29번과 같은 수준)

| 검증 | 조합 | CV | C 대비 | C보다 높은 fold | 신장 그룹 밖 | 신장 그룹 안 혼동 | 뇌 그룹 밖 | 뇌 그룹 안 혼동 | 나머지 22개 평균 F1 | 끌려온 환자 |
| :--- | :--- | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: | :-: |
| 공통 CV | C | 0.4557 | — | — | 32.7% | 49.0% | 24.6% | 38.4% | 0.4855 | 234 |
| 공통 CV | 31a 큰 그룹 + C 비율 | 0.4481 | −0.0075 | 1/5 | **44.2%** | 41.5% | 26.8% | 37.7% | 0.4781 | **109** |
| 공통 CV | 31b 큰 그룹 + 그룹 안 모델 | 0.4460 | −0.0096 | 1/5 | 44.2% | 42.0% | 26.8% | 40.6% | 0.4791 | 104 |
| 공통 CV | **31c 큰 그룹(26개 기준 가중치) + C 비율** | **0.4562** | +0.0006 | **3/5** | 33.6% | 48.4% | 24.8% | 38.3% | 0.4863 | 224 |
| 같은 프로필 묶음 CV | C | 0.4685 | — | — | 40.4% | 26.7% | 25.1% | 30.1% | 0.4799 | 232 |
| 같은 프로필 묶음 CV | 31a | 0.4666 | −0.0018 | 1/5 | 48.5% | 23.8% | 28.7% | 29.1% | 0.4799 | 105 |
| 같은 프로필 묶음 CV | 31b | 0.4672 | −0.0013 | 2/5 | 48.4% | 24.1% | 29.1% | 28.8% | 0.4807 | 104 |
| 같은 프로필 묶음 CV | **31c** | **0.4700** | +0.0016 | **2/5** | 39.3% | 27.3% | 26.1% | 29.7% | 0.4810 | 222 |

- 끌려온 환자 = 실제로는 다른 암종인데 신장·뇌로 예측된 환자 (이 노트북의 C 시드 42 기준 234명, 선영님 확인 254명은 다른 설정 기준일 수 있음)

**네 클래스 F1 (C → 31a → 31b → 31c)**

| 클래스 | 공통 CV | 같은 프로필 묶음 CV |
| :--- | :-: | :-: |
| KIPAN | 0.206 → 0.163 → 0.162 → 0.201 | 0.323 → 0.272 → 0.287 → 0.322 |
| KIRC | 0.180 → 0.190 → 0.178 → 0.184 | 0.418 → 0.434 → 0.416 → 0.426 |
| GBMLGG | 0.324 → 0.314 → 0.313 → 0.324 | 0.389 → 0.368 → 0.384 → 0.386 |
| LGG | 0.479 → 0.484 → 0.420 → 0.483 | 0.578 → 0.584 → 0.568 → 0.579 |
| 나머지 22개 평균 | 0.485 → 0.478 → 0.479 → 0.486 | 0.480 → 0.480 → 0.481 → 0.481 |

**해석**
- **31a·31b**: 큰 그룹으로 묶으니 가설과 반대로 그룹 밖 오답이 늘었음 (신장 32.7% → 44.2%), 끌려온 환자는 234 → 109명으로 감소
  - 원인: balanced weight가 클래스마다 같은 무게를 줘서, 신장 한 그룹이 되면 신장 전체 무게가 C의 약 절반 → 모델이 신장을 덜 예측
- **31c (가중치를 26개 기준으로 되돌림)**: 이 문제는 해결됨 → 신장 그룹 밖 33.6%, 끌려온 환자 224명으로 C와 거의 같은 수준으로 돌아옴
  - 네 클래스 F1도 31a에서 떨어졌던 KIPAN이 C 수준으로 회복 (0.163 → 0.201)
  - 다만 **C를 넘지는 못함**: 공통 CV +0.0006(3/5), 같은 프로필 묶음 CV +0.0016(2/5) → 무게가 C와 같아지니 결과도 C와 거의 같아짐
  - 신장 그룹 밖 오답은 C보다 줄지 않음 (공통 CV 32.7% → 33.6%, 묶음 CV 40.4% → 39.3%) → "신장 신호를 한곳에 모으는" 효과는 거의 없음
- 그룹 안 모델(31b)은 31a보다 나아지지 않음 → 29번과 같은 흐름
- 참고: 같은 프로필 묶음 CV에서는 C의 신장 그룹 안 혼동이 26.7%로 공통 CV(49.0%)보다 훨씬 낮음 → 공통 CV에서 보이던 KIRC↔KIPAN 혼동의 상당 부분은 같은 프로필이 갈라져서 생긴 것

## 결론
- **H31a 기각**: 큰 그룹 + C 비율은 C보다 공통 CV 1/5, 묶음 CV 1/5 fold (balanced weight로 신장 무게가 줄어 그룹 밖 오답 증가)
- **H31b 기각**: 큰 그룹 + 그룹 안 모델도 공통 CV 1/5, 묶음 CV 2/5 fold
- **H31c 기각**: 가중치를 26개 기준으로 되돌리면 31a의 문제는 사라지지만 C와 거의 같음 (공통 CV 3/5, 묶음 CV 2/5) → 큰 그룹으로 묶어 학습하는 것 자체의 이득은 확인되지 않음
- 22b XGB는 이미 KIRC·KIPAN을 따로 학습하면서도 "신장" 공통 신호를 충분히 배우고 있는 것으로 보임 → 제출 후보에서 제외, C 계열 유지
- 다른 암종이 끌려오는 문제는 세 방식 모두 C보다 늘지 않았음, 22개 암종 F1도 거의 같음
- 시드 평균은 채택 기준(두 검증 모두 4/5)을 넘는 설정이 없어 실행하지 않음
- 결과물 (데스크탑에서 실행하면 저장, 기준 미달이라 기록용): `blend_hier31a_0.7_LR0.3`, `blend_hier31b_0.7_LR0.3`, `blend_hier31c_0.7_LR0.3`
